# 创建自定义偏置

## 为什么需要自定义偏置？

虽然NSGABLACK提供了多种内置偏置，但实际应用中的问题千差万别。自定义偏置可以让您：

1. **融入领域知识** - 将专业经验转化为优化策略
2. **解决特定问题** - 针对特殊约束或目标设计偏置
3. **优化性能** - 为特定算法或问题定制偏置

In [ ]:
# 导入必要的库
import numpy as np
import matplotlib.pyplot as plt

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("环境准备完成")

# 定义偏置基类（简化版）
class BiasBase:
    """偏置基类 - 所有偏置的父类"""
    
    def __init__(self, name="CustomBias"):
        self.name = name
        self.initialized = False
    
    def initialize(self, problem):
        """初始化偏置"""
        self.initialized = True
        print(f"偏置 '{self.name}' 已初始化")
    
    def apply(self, population, problem, generation):
        """应用偏置 - 核心方法"""
        if not self.initialized:
            print(f"警告：偏置 '{self.name}' 未初始化")
        return population

print("偏置基类已定义")
print("核心方法：")
print("  - __init__() - 构造函数")
print("  - initialize() - 初始化偏置")
print("  - apply() - 应用偏置（核心）")

## 示例1：目标点偏置

将种群向指定目标点偏置。

In [ ]:
# 目标点偏置
class TargetPointBias(BiasBase):
    """目标点偏置 - 将种群向指定目标点偏置"""
    
    def __init__(self, target_point, bias_strength=0.1):
        super().__init__("TargetPointBias")
        self.target_point = np.array(target_point)
        self.bias_strength = bias_strength
        print(f"创建目标点偏置")
        print(f"  目标点: {self.target_point}")
        print(f"  偏置强度: {self.bias_strength}")
    
    def apply(self, population, problem, generation):
        """应用偏置"""
        biased_population = []
        
        for individual in population:
            ind_array = np.array(individual)
            direction = self.target_point - ind_array
            biased_ind = ind_array + self.bias_strength * direction
            biased_population.append(biased_ind.tolist())
        
        return biased_population

# 测试目标点偏置
print("测试目标点偏置：")
print("-"*40)

# 创建偏置
target_bias = TargetPointBias(target_point=[1, 1], bias_strength=0.3)

# 创建测试种群
test_pop = [
    [5, 5],
    [0, 0],
    [-2, 3],
    [3, -1]
]

print("\n原始种群：")
for i, ind in enumerate(test_pop):
    print(f"  个体{i}: {ind}")

# 应用偏置
biased_pop = target_bias.apply(test_pop, None, 1)

print("\n偏置后种群：")
for i, ind in enumerate(biased_pop):
    print(f"  个体{i}: {ind}")

# 可视化偏置效果
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 原始种群
ax1 = axes[0]
pop_array = np.array(test_pop)
ax1.scatter(pop_array[:, 0], pop_array[:, 1], s=100, alpha=0.7)
ax1.scatter(1, 1, s=200, c='red', marker='*', label='目标点')
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_title('原始种群')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 偏置后种群
ax2 = axes[1]
biased_array = np.array(biased_pop)
ax2.scatter(pop_array[:, 0], pop_array[:, 1], s=50, alpha=0.3, c='gray', label='原始')
ax2.scatter(biased_array[:, 0], biased_array[:, 1], s=100, alpha=0.7, label='偏置后')
ax2.scatter(1, 1, s=200, c='red', marker='*', label='目标点')

# 绘制偏置方向箭头
for i in range(len(test_pop)):
    ax2.arrow(pop_array[i, 0], pop_array[i, 1],
              biased_array[i, 0] - pop_array[i, 0],
              biased_array[i, 1] - pop_array[i, 1],
              head_width=0.1, head_length=0.05,
              fc='blue', ec='blue', alpha=0.5)

ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_title('目标点偏置效果')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 示例2：边界约束偏置

将解限制在指定的边界内。

In [ ]:
# 边界约束偏置
class BoundaryBias(BiasBase):
    """边界约束偏置 - 将解限制在指定范围内"""
    
    def __init__(self, bounds=(-1, 1)):
        super().__init__("BoundaryBias")
        self.lower, self.upper = bounds
        print(f"创建边界偏置 [{self.lower}, {self.upper}]")
    
    def apply(self, population, problem, generation):
        """应用边界偏置"""
        biased_population = []
        
        for individual in population:
            # 使用clip函数将值限制在边界内
            biased_ind = np.clip(individual, self.lower, self.upper)
            biased_population.append(biased_ind.tolist())
        
        return biased_population

# 测试边界偏置
print("\n测试边界偏置：")
print("-"*40)

# 创建边界偏置
boundary_bias = BoundaryBias(bounds=(-2, 2))

# 创建超出边界的测试种群
unbounded_pop = [
    [3.5, 1.2],   # 超出上界
    [-3.0, 0.5],  # 超出下界
    [0.5, 2.8],   # 部分超出
    [0.0, 0.0],   # 在界内
    [-1.5, -1.5]  # 在界内
]

print("\n无约束种群：")
for i, ind in enumerate(unbounded_pop):
    print(f"  个体{i}: {ind}")

# 应用边界偏置
constrained_pop = boundary_bias.apply(unbounded_pop, None, 1)

print("\n约束后种群：")
for i, ind in enumerate(constrained_pop):
    print(f"  个体{i}: {ind}")

# 可视化边界偏置效果
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 无约束种群
ax1 = axes[0]
unbounded_array = np.array(unbounded_pop)
ax1.scatter(unbounded_array[:, 0], unbounded_array[:, 1], s=100, alpha=0.7)
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_title('无约束种群')
ax1.grid(True, alpha=0.3)
ax1.set_xlim([-4, 4])
ax1.set_ylim([-4, 4])

# 约束后种群
ax2 = axes[1]
constrained_array = np.array(constrained_pop)
ax2.scatter(unbounded_array[:, 0], unbounded_array[:, 1], 
           s=50, alpha=0.3, c='gray', label='原始')
ax2.scatter(constrained_array[:, 0], constrained_array[:, 1], 
           s=100, alpha=0.7, label='约束后')

# 绘制边界框
ax2.plot([-2, -2, 2, 2, -2], [-2, 2, 2, -2, -2], 'r-', linewidth=2, label='边界')

ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_title('边界约束效果')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xlim([-4, 4])
ax2.set_ylim([-4, 4])

plt.tight_layout()
plt.show()

## 示例3：自适应强度偏置

根据优化进展自动调整偏置强度。

In [ ]:
# 自适应强度偏置
class AdaptiveStrengthBias(BiasBase):
    """自适应强度偏置 - 根据改进情况动态调整偏置强度"""
    
    def __init__(self, initial_strength=0.3, min_strength=0.01, max_strength=0.5):
        super().__init__("AdaptiveStrengthBias")
        self.initial_strength = initial_strength
        self.min_strength = min_strength
        self.max_strength = max_strength
        
        self.current_strength = initial_strength
        self.best_objective = float('inf')
        self.no_improvement_count = 0
        
        print(f"创建自适应强度偏置")
        print(f"  初始强度: {initial_strength}")
        print(f"  强度范围: [{min_strength}, {max_strength}]")
    
    def apply(self, population, problem, generation):
        """应用偏置 - 向原点偏置"""
        target = np.zeros(len(population[0]))
        biased_population = []
        
        for individual in population:
            ind_array = np.array(individual)
            direction = target - ind_array
            biased_ind = ind_array + self.current_strength * direction
            biased_population.append(biased_ind.tolist())
        
        return biased_population
    
    def update(self, population, objectives, problem):
        """更新偏置强度"""
        current_best = min(objectives)
        
        # 检查是否有改进
        if current_best < self.best_objective:
            self.best_objective = current_best
            self.no_improvement_count = 0
            # 有改进时，稍微增加偏置强度
            self.current_strength = min(self.current_strength * 1.05, self.max_strength)
        else:
            self.no_improvement_count += 1
            # 无改进时，降低偏置强度
            self.current_strength = max(self.current_strength * 0.95, self.min_strength)

# 模拟优化过程
print("\n测试自适应强度偏置：")
print("-"*40)

adaptive_bias = AdaptiveStrengthBias(initial_strength=0.3)

# 模拟50代优化
best_values = []
strength_history = []

population = [[np.random.uniform(-5, 5), np.random.uniform(-5, 5)] for _ in range(20)]

for gen in range(50):
    # 应用偏置
    population = adaptive_bias.apply(population, None, gen)
    
    # 模拟评估
    objectives = [np.linalg.norm(np.array(ind)) for ind in population]
    best_values.append(min(objectives))
    strength_history.append(adaptive_bias.current_strength)
    
    # 更新偏置
    adaptive_bias.update(population, objectives, None)

# 可视化自适应过程
fig, axes = plt.subplots(2, 1, figsize=(10, 8))

# 目标值变化
ax1 = axes[0]
ax1.plot(best_values, 'b-', linewidth=2)
ax1.set_ylabel('最佳目标值')
ax1.set_title('优化过程')
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# 偏置强度变化
ax2 = axes[1]
ax2.plot(strength_history, 'r-', linewidth=2)
ax2.set_xlabel('代数')
ax2.set_ylabel('偏置强度')
ax2.set_title('自适应偏置强度变化')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n最终统计：")
print(f"  初始最佳值: {best_values[0]:.4f}")
print(f"  最终最佳值: {best_values[-1]:.4f}")
print(f"  改进比例: {((best_values[0] - best_values[-1])/best_values[0]*100):.2f}%")
print(f"  初始偏置强度: {strength_history[0]:.4f}")
print(f"  最终偏置强度: {strength_history[-1]:.4f}")

## 示例4：偏置组合

多个偏置可以组合使用。

In [ ]:
# 偏置组合器
class BiasComposer:
    """偏置组合器 - 组合多个偏置"""
    
    def __init__(self, biases):
        self.biases = biases
        print(f"创建偏置组合器，包含 {len(biases)} 个偏置")
    
    def apply(self, population, problem, generation):
        """顺序应用所有偏置"""
        result = population
        for bias in self.biases:
            result = bias.apply(result, problem, generation)
        return result

# 创建组合偏置
print("\n测试偏置组合：")
print("-"*40)

# 创建多种偏置
target_bias = TargetPointBias(target_point=[0, 0], bias_strength=0.2)
boundary_bias = BoundaryBias(bounds=(-3, 3))

# 创建组合
composer = BiasComposer([boundary_bias, target_bias])

# 测试种群
test_pop = [
    [5, 5],      # 超出边界，远离目标
    [-4, 3],     # 超出边界
    [2, -2],     # 在边界内
    [-1, -1]     # 在边界内，接近目标
]

print("\n原始种群：")
for i, ind in enumerate(test_pop):
    print(f"  个体{i}: {ind}")

# 逐步显示效果
step1 = boundary_bias.apply(test_pop, None, 1)
print(f"\n1. 边界偏置后: {step1}")

step2 = target_bias.apply(step1, None, 1)
print(f"2. 目标偏置后: {step2}")

# 使用组合器
combined = composer.apply(test_pop, None, 1)
print(f"\n组合偏置结果: {combined}")

# 可视化组合效果
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 原始
ax1 = axes[0]
orig_array = np.array(test_pop)
ax1.scatter(orig_array[:, 0], orig_array[:, 1], s=100, alpha=0.7)
ax1.set_title('原始种群')
ax1.grid(True, alpha=0.3)

# 边界偏置后
ax2 = axes[1]
step1_array = np.array(step1)
ax2.scatter(orig_array[:, 0], orig_array[:, 1], s=50, alpha=0.3, c='gray')
ax2.scatter(step1_array[:, 0], step1_array[:, 1], s=100, alpha=0.7, c='orange')
ax2.plot([-3, -3, 3, 3, -3], [-3, 3, 3, -3, -3], 'r--', alpha=0.5)
ax2.set_title('边界偏置后')
ax2.grid(True, alpha=0.3)

# 最终组合
ax3 = axes[2]
combined_array = np.array(combined)
ax3.scatter(combined_array[:, 0], combined_array[:, 1], s=100, alpha=0.7, c='green')
ax3.scatter(0, 0, s=200, c='red', marker='*')
ax3.set_title('最终组合偏置')
ax3.grid(True, alpha=0.3)

# 设置相同的坐标范围
for ax in axes:
    ax.set_xlim([-6, 6])
    ax.set_ylim([-6, 6])
    ax.set_xlabel('X')
    ax.set_ylabel('Y')

plt.tight_layout()
plt.show()

## 总结

### 创建偏置的步骤

1. **继承基类** - `class YourBias(BiasBase)`
2. **实现构造函数** - 初始化参数
3. **实现apply方法** - 核心偏置逻辑
4. **（可选）实现update方法** - 自适应更新

### 开发建议

- **单一职责** - 每个偏置专注一个功能
- **参数化** - 通过参数控制行为
- **可测试** - 创建测试用例
- **文档化** - 说明偏置作用

### 下一步

现在您已经掌握了如何创建自定义偏置。接下来学习如何将偏置应用到具体的优化算法中！